<a href="https://colab.research.google.com/github/alaaguedda/medical_report_summarization_project/blob/enhanced/medical_summerizer_trained.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [32]:
!pip install kaggle
from google.colab import files
files.upload()
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d aminexdr/bhc-mimic-iv-summary
!unzip bhc-mimic-iv-summary.zip


Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/aminexdr/bhc-mimic-iv-summary
License(s): unknown
bhc-mimic-iv-summary.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  bhc-mimic-iv-summary.zip
replace BHC_MIMIC-IV.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: BHC_MIMIC-IV.csv        


In [33]:
import pandas as pd

df = pd.read_csv("BHC_MIMIC-IV.csv")


In [34]:
import pandas as pd

# 1. Basic Cleaning & Sampling
df = df[['input', 'target']].dropna()
df = df.sample(frac=1, random_state=42).iloc[:15000]

# 2. Helper function to slice by word count
def limit_words(text, limit):
    if not isinstance(text, str):
        return ""
    words = text.split()
    return " ".join(words[:limit])

# 3. Apply the limits
df['input'] = df['input'].apply(lambda x: limit_words(x, 300))   # more context
df['target'] = df['target'].apply(lambda x: limit_words(x, 80))  # still compressed

print(f"Dataframe processed. Shape: {df.shape}")

Dataframe processed. Shape: (20000, 2)


In [35]:
df.head(

)

,input,target
267505,write a discharge summary: History of Present ...,Patient arrived on the unit intubated and seda...
180880,generate a brief hospital summary: Chief Compl...,"AP: year old female with ho CLL, PAF, not on c..."
252848,create a summary based on the following inform...,Mrs. is a yr old female presenting with new ab...
112160,write a discharge summary: Chief Complaint: Ch...,Patient had recurring chest pain consistent wi...
99808,create a summary based on the following inform...,"Mr. is a M w ho CAD , atrial fibrillation sp c..."


In [5]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Define the path where you want to save it
# Change 'my_folder' to whatever folder exists in your Drive
save_path = '/content/drive/MyDrive/cleaned_summary_data.csv'

# Save the dataframe
df.to_csv(save_path, index=False)

print(f"File successfully saved to: {save_path}")

Mounted at /content/drive
File successfully saved to: /content/drive/MyDrive/cleaned_summary_data.csv


In [31]:
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')
df = pd.read_csv('/content/drive/MyDrive/cleaned_summary_data.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
df.head(
)

,input,target
267505,write a discharge summary:\nHistory of Present...,Patient arrived on the unit intubated and seda...
180880,generate a brief hospital summary:\nChief Comp...,"AP: year old female with ho CLL, PAF, not on ..."
252848,create a summary based on the following inform...,Mrs. is a yr old female presenting with new ...
112160,write a discharge summary:\nChief Complaint: C...,Patient had recurring chest pain consistent wi...
99808,create a summary based on the following inform...,"Mr. is a M w ho CAD , atrial fibrillation sp..."


In [ ]:
df.isnull().sum()
df = df.dropna()

In [37]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

In [ ]:
!pip install transformers datasets evaluate rouge-score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.4 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=30c0a8fce2a4e4be466b1e1e60856ceb792a40aa30144cc431faf59ef1a68b9f
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [38]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

model_name = "google/flan-t5-base"

tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [42]:
def preprocess_function(examples):
    # Guarantee the prefix is always present and consistent
    inputs = [
        "summarize: " + str(text) if not str(text).startswith("summarize:") else str(text)
        for text in examples["input"]
    ]

    model_inputs = tokenizer(
        inputs,
        max_length=256,
        truncation=True,
        padding=False
    )

    # ✅ Just tokenize labels directly — as_target_tokenizer() is deprecated/removed
    labels = tokenizer(
        text_target=examples["target"],   # use text_target= instead
        max_length=64,
        truncation=True,
        padding=False
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [43]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    padding=True,
    pad_to_multiple_of=8   # helps with fp16 performance on T4
)

In [44]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

train_dataset = train_dataset.map(preprocess_function, batched=True)
val_dataset = val_dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [54]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=3e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=2,
    eval_accumulation_steps=10,
    gradient_accumulation_steps=2,
    load_best_model_at_end=True,
    save_strategy="epoch",          # needed for early stopping
    metric_for_best_model="eval_ROUGE-L",   # ✅ add eval_ prefix explicitly
    greater_is_better=True,
    num_train_epochs=6,
    weight_decay=0.01,
    fp16=True,
    predict_with_generate=False,  # Now this will work!
    generation_max_length=64,
    generation_num_beams=4,
    logging_steps=200,
    report_to="none"
)

In [55]:
import numpy as np
import evaluate

rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    if isinstance(predictions, tuple):
        predictions = predictions[0]

    if predictions.ndim == 3:
        predictions = np.argmax(predictions, axis=-1)

    # Fix: clip predictions to valid token id range
    vocab_size = tokenizer.vocab_size
    predictions = np.clip(predictions, 0, vocab_size - 1)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    # Fix: also clip labels just in case
    labels = np.clip(labels, 0, vocab_size - 1)

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [label.strip() for label in decoded_labels]

    rouge_result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels
    )

    return {
        "ROUGE-1": rouge_result["rouge1"],
        "ROUGE-2": rouge_result["rouge2"],
        "ROUGE-L": rouge_result["rougeL"],
    }

In [56]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,                          # ✅ must be here
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss


In [14]:
!pip install evaluate rouge_score absl-py

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.2 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=88fe4d2073e24749b7248ca9131a4812aac2f66fe857987603aa2ca61cb3591d
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [ ]:
import torch
import gc

# 1. Clear Python garbage collector
gc.collect()

# 2. Clear NVIDIA cache
torch.cuda.empty_cache()



In [16]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,  # Fix: renamed from 'tokenizer'
    compute_metrics=compute_metrics
)

In [17]:
results = trainer.evaluate()

In [18]:
print(results)

{'eval_loss': 2.023879051208496, 'eval_model_preparation_time': 0.0168, 'eval_ROUGE-1': 0.39373792638580385, 'eval_ROUGE-2': 0.23514057938831684, 'eval_ROUGE-L': 0.34341088414787313, 'eval_runtime': 1028.684, 'eval_samples_per_second': 1.944, 'eval_steps_per_second': 0.972}


In [27]:
import torch
import textwrap

def generate_and_compare(sample_input, reference_summary=None,
                         input_preview_chars=500,
                         max_input_length=512,
                         max_target_length=128,
                         num_beams=4):
    """
    Generates summary and prints structured comparison.

    Parameters
    ----------
    sample_input : str
        The original medical report text.
    reference_summary : str, optional
        The ground truth summary (if available).
    input_preview_chars : int
        Number of characters from original input to display.
    max_input_length : int
        Token truncation length for input.
    max_target_length : int
        Maximum generation length.
    num_beams : int
        Beam search width.
    """

    device = model.device
    model.eval()

    # ---- Prepare Input ----
    input_text = "summarize: " + sample_input if not sample_input.startswith("summarize:") else sample_input

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=max_input_length,
        truncation=True
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    # ---- Generate ----
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_target_length,
            num_beams=num_beams,
            early_stopping=True
        )

    generated_summary = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    # ---- Pretty Printing Section ----
    print("\n" + "="*80)
    print("📝 ORIGINAL INPUT (Preview)")
    print("="*80)
    print(textwrap.fill(sample_input[:input_preview_chars], width=100))

    print("\n" + "-"*80)
    print("🤖 GENERATED SUMMARY")
    print("-"*80)
    print(textwrap.fill(generated_summary, width=100))


    print("="*80 + "\n")

    return generated_summary

In [36]:
generate_and_compare(
    sample_input=df['input'].iloc[17],
)


📝 ORIGINAL INPUT (Preview)
summarize: Chief Complaint: dizziness History of Present Illness: is a right-handed man with past
medical history of L MCA stroke in secondary to cocaine use, depression with psychotic features and
dyslipidemia, who presents from group home with complaints of vertigo and an occipital head strike.
The history is extremely limited as pt has limited and is aphasic but it appears that he developed
sudden room-spinning vertigo yesterday, and has continued to experience this even now at rest. It is
no

--------------------------------------------------------------------------------
🤖 GENERATED SUMMARY
--------------------------------------------------------------------------------
is a right-handed man with past medical history of L MCA stroke . he presents from group home with
complaints of vertigo and an occipital head strike . he does endorse difficulty walking but does not
specify further .



'is a right-handed man with past medical history of L MCA stroke . he presents from group home with complaints of vertigo and an occipital head strike . he does endorse difficulty walking but does not specify further .'

In [29]:
test_report = """
	generate a brief hospital summary: Chief Complaint: diarrhea, low blood pressure History of Present Illness: year old female with ho CLL, PAF, not on coumadin, who presents with chief complaint of diarrhea and abdominal pain x 3 days. Denies associated nausea, vomiting. Reports trying a tea and toast diet without improvement of symptoms. She has chronic abdominal pain secondary to CLL, however reports worse pain in left lower abdomen, compared to baseline. No fever, chills, night sweats. No lightheadedness, dizziness, cp or palpitations. No brbpr or melanotic stools. Seen in clinic today, BP 9858, HR 123 . Referred to ER."""

generate_and_compare(sample_input=test_report)


📝 ORIGINAL INPUT (Preview)
         generate a brief hospital summary: Chief Complaint: diarrhea, low blood pressure History of
Present Illness: year old female with ho CLL, PAF, not on coumadin, who presents with chief
complaint of diarrhea and abdominal pain x 3 days. Denies associated nausea, vomiting. Reports
trying a tea and toast diet without improvement of symptoms. She has chronic abdominal pain
secondary to CLL, however reports worse pain in left lower abdomen, compared to baseline. No fever,
chills, night sweats. No li

--------------------------------------------------------------------------------
🤖 GENERATED SUMMARY
--------------------------------------------------------------------------------
year old female with ho CLL, PAF, not on coumadin, who presents with chief complaint of diarrhea and
abdominal pain x 3 days.



'year old female with ho CLL, PAF, not on coumadin, who presents with chief complaint of diarrhea and abdominal pain x 3 days.'